# Section 09 — Plugins
## Lecture 9.2 — Meet the Full Capstone: The Autonomous Event Genie

In this notebook you bring back the Autonomous Event Genie, the ten-agent, five-workflow capstone system you built earlier in the course. You will run it exactly as it stands today: zero callbacks, zero Plugins, nothing wired in for visibility. Watch it work end to end, and pay close attention to how little you can tell about what happened inside it once it finishes.

## Cell 2 — Install the ADK

This notebook uses the Google Agent Development Kit (ADK) to build and run a multi-agent system. The cell below installs the exact version of `google-adk` used throughout this notebook. If the package is already present in this Colab runtime, the install completes almost instantly and moves on.

In [ ]:
# Pinned for reproducibility. To use the latest version,
# run: pip install google-adk
# Or substitute your preferred version below.
!pip install google-adk==2.6.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 14.9 MB/s eta 0:00:00


## Cell 3 — Configure the Gemini API Key

Every agent in this notebook calls a Gemini model, so you need a Gemini API key before anything else will run. This notebook uses Google Colab's built-in Secrets manager, which keeps the key out of the notebook's saved text.

**To add your key as a Colab secret:**
1. Click the key icon in the left sidebar of this Colab notebook.
2. Click **Add new secret**.
3. Name it `GOOGLE_API_KEY`.
4. Paste in your key from Google AI Studio.
5. Toggle **Notebook access** on so this notebook can read it.

**Running locally instead of in Colab?** Set `GOOGLE_API_KEY` as an environment variable in your terminal before starting Jupyter, rather than using `userdata.get`.

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## Cell 4 — Declare the Model Name

Every agent defined later in this notebook references `MODEL_NAME` instead of a hardcoded model string. Change the value in this one cell and every agent in the notebook picks up the new model automatically.

| Variable | Value | Purpose |
|---|---|---|
| `MODEL_NAME` | `"gemini-2.5-flash"` | The model used by every agent defined below |

In [ ]:
# See latest models at: https://ai.google.dev/gemini-api/docs/models
MODEL_NAME = "gemini-3.7-flash"

## Cell 5 — Define the Custom Tools

The Autonomous Event Genie relies on six plain Python functions instead of one monolithic tool. Each one gives a single agent a narrow, well-defined capability: writing to session state, summing costs, or breaking out of a loop. `GUEST_DATABASE` is a simple in-memory list standing in for a real datastore, and `COMPLETION_PHRASE` is the exact string `accountant_agent` writes when a plan is on budget — which `cost_cutter_agent` checks for before calling `exit_loop`.

| Tool | Used by | Purpose |
|---|---|---|
| `update_session_state` | `intake_agent` | Writes `event_type`, `city`, and `budget` to session state |
| `add_guest`, `get_guest_list` | `guest_management_agent` | Manage the in-memory guest list |
| `sum_costs` | `accountant_agent` | Totals a list of costs |
| `exit_loop` | `cost_cutter_agent` | Sets `tool_context.actions.escalate = True` to stop the budget loop |
| `send_mock_email` | `master_orchestrator_agent` | Simulates sending the drafted announcement email |

In [ ]:
from google.adk.tools import ToolContext

GUEST_DATABASE = []

COMPLETION_PHRASE = "The plan is within the budget."


def add_guest(name: str, email: str) -> dict:
    """Adds a guest to the guest database.

    Args:
        name: The full name of the guest.
        email: The guest's email address.

    Returns:
        A dictionary confirming the guest was added.
    """
    GUEST_DATABASE.append({"name": name, "email": email})
    return {"status": "success", "guest": name}


def get_guest_list(tool_context: ToolContext) -> dict:
    """Retrieves the current guest list and writes it to session state.

    Args:
        tool_context: The tool context, used to write to session state.

    Returns:
        A dictionary containing the current guest list.
    """
    tool_context.state["guest_list"] = GUEST_DATABASE
    return {"guest_list": GUEST_DATABASE}


def sum_costs(costs: list[float]) -> float:
    """Sums a list of costs.

    Args:
        costs: A list of numeric cost values.

    Returns:
        The total of the provided costs.
    """
    return sum(costs)


def exit_loop(tool_context: ToolContext) -> dict:
    """Signals the budget refinement loop to stop iterating.

    Args:
        tool_context: The tool context, used to set the escalate action.

    Returns:
        A dictionary confirming the loop will exit.
    """
    tool_context.actions.escalate = True
    return {"status": "loop_exited"}


def update_session_state(
    tool_context: ToolContext,
    event_type: str,
    city: str,
    budget: float,
) -> dict:
    """Writes the extracted event details to session state.

    Args:
        tool_context: The tool context, used to write to session state.
        event_type: The type of event being planned.
        city: The city where the event will take place.
        budget: The total budget available for the event.

    Returns:
        A dictionary confirming the values that were written.
    """
    tool_context.state["event_type"] = event_type
    tool_context.state["city"] = city
    tool_context.state["budget"] = budget
    return {"event_type": event_type, "city": city, "budget": budget}


def send_mock_email(tool_context: ToolContext) -> dict:
    """Simulates sending the drafted announcement email to every guest.

    Args:
        tool_context: The tool context, used to read the drafted email.

    Returns:
        A dictionary summarizing the simulated send.
    """
    email_draft = tool_context.state.get("email_draft", {})
    recipients = [guest["email"] for guest in GUEST_DATABASE]
    return {
        "status": "sent",
        "subject": email_draft.get("subject", ""),
        "recipient_count": len(recipients),
    }

## Cell 6 — Wrap Google Search as a Sub-Agent Tool

`google_search` is a built-in ADK tool, but ADK only allows it on an agent that has no other tools attached. `venue_scout_agent`, `catering_scout_agent`, and `entertainment_scout_agent` attach `google_search` directly, since search is their only tool. `cost_cutter_agent` needs both search and `exit_loop` at the same time, so instead of attaching `google_search` directly, it calls a small dedicated search agent through an `AgentTool` wrapper. The wrapped agent does nothing but search — that sidesteps the one-tool restriction without changing what the search itself returns.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

google_search_agent = Agent(
    name="Google_Search_Agent",
    model=MODEL_NAME,
    instruction="You are just a wrapper for the Google Search tool.",
    tools=[google_search]
)
google_search_tool = AgentTool(agent=google_search_agent)

## Cell 7 — Define the Ten Specialist Agents

This is the full roster of specialist agents that make up the Event Genie. Every agent below is deliberately narrow — it does one job, reads a small number of state keys, and in most cases writes its result back to state through `output_key` so the next agent in the workflow can pick it up. Instruction strings reference state with single curly braces, for example `{event_type}`, which ADK resolves against the session state before the prompt ever reaches the model.

| Agent | Role | Tools | `output_key` |
|---|---|---|---|
| `intake_agent` | Extracts `event_type`, `city`, `budget` from the user's request | `update_session_state` | — |
| `guest_management_agent` | Manages the guest list | `add_guest`, `get_guest_list` | — |
| `venue_scout_agent` | Scouts 3 venues with costs | `google_search` | `venue_options` |
| `catering_scout_agent` | Scouts 3 caterers with costs | `google_search` | `catering_options` |
| `entertainment_scout_agent` | Scouts 3 entertainment options with costs | `google_search` | `entertainment_options` |
| `initial_plan_synthesizer_agent` | Combines the three scouts' results into one plan | — | `current_plan` |
| `accountant_agent` | Picks the cheapest option per category, checks against budget | `sum_costs` | `evaluation` |
| `cost_cutter_agent` | Finds a cheaper alternative if the plan is over budget | `google_search_tool`, `exit_loop` | `current_plan` |
| `communications_agent` | Drafts the announcement email | — | `email_draft` |
| `final_report_agent` | Compiles the final markdown report | — | — |

`communications_agent` carries a hard structural guarantee via `output_schema=EmailDraft`: its final output is validated against the `EmailDraft` model defined just below, so `state["email_draft"]` always has a `subject` and a `body`, no matter how the model phrases its response.

In [ ]:
from pydantic import BaseModel, Field


class EmailDraft(BaseModel):
    subject: str = Field(description="The compelling subject line for the email.")
    body: str = Field(description="The full, well-formatted body of the email.")


intake_agent = Agent(
    name="intake_agent",
    model=MODEL_NAME,
    instruction="From the user's query, identify the event type, city, and budget. Then call update_session_state.",
    tools=[update_session_state]
)

guest_management_agent = Agent(
    name="guest_management_agent",
    model=MODEL_NAME,
    instruction="You are a guest management assistant. Use add_guest and get_guest_list tools.",
    tools=[add_guest, get_guest_list]
)

venue_scout_agent = Agent(
    name="venue_scout_agent",
    model=MODEL_NAME,
    instruction='Find 3 venues for {event_type} in {city} with costs. Output JSON: {"venues": [...]}',
    tools=[google_search],
    output_key="venue_options"
)

catering_scout_agent = Agent(
    name="catering_scout_agent",
    model=MODEL_NAME,
    instruction='Find 3 caterers for {event_type} in {city} with costs. Output JSON: {"caterers": [...]}',
    tools=[google_search],
    output_key="catering_options"
)

entertainment_scout_agent = Agent(
    name="entertainment_scout_agent",
    model=MODEL_NAME,
    instruction='Find 3 entertainment options for {event_type} in {city} with costs. Output JSON: {"entertainment": [...]}',
    tools=[google_search],
    output_key="entertainment_options"
)

initial_plan_synthesizer_agent = Agent(
    name="initial_plan_synthesizer_agent",
    model=MODEL_NAME,
    instruction="Combine {venue_options}, {catering_options}, {entertainment_options} into one JSON with keys venues, caterers, entertainment.",
    output_key="current_plan"
)

accountant_agent = Agent(
    name="accountant_agent",
    model=MODEL_NAME,
    tools=[sum_costs],
    instruction="Select cheapest option from each category in {current_plan}. Use sum_costs. If total > {budget}, output JSON with critique and cheapest_plan. Else output JSON with completion phrase and cheapest_plan.",
    output_key="evaluation"
)

cost_cutter_agent = Agent(
    name="cost_cutter_agent",
    model=MODEL_NAME,
    tools=[google_search_tool, exit_loop],
    instruction="Check {evaluation} critique. If completion phrase, call exit_loop. Else find cheaper alternative for the flagged item. Output updated plan JSON.",
    output_key="current_plan"
)

communications_agent = Agent(
    name="communications_agent",
    model=MODEL_NAME,
    instruction='Draft announcement email from {current_plan}. Output raw JSON only: {"subject": ..., "body": ...}',
    output_key="email_draft",
    output_schema=EmailDraft
)

final_report_agent = Agent(
    name="final_report_agent",
    model=MODEL_NAME,
    instruction="Compile final event plan markdown report using {current_plan}, {venue_options}, {catering_options}, {entertainment_options}, {budget}. Include executive summary, approved plan, alternatives, next steps."
)

## Cell 8 — Assemble the Five Workflow Agents

Individually, the ten specialist agents cannot coordinate themselves. The five workflow agents below turn them into the actual Event Genie pipeline: `ParallelAgent` runs the three scouts that don't depend on each other at the same time, `SequentialAgent` runs steps that must happen in a fixed order, and `LoopAgent` drives the iterate-until-on-budget refinement cycle. `budget_refinement_loop` caps out at `max_iterations=3` as a safety valve independent of `exit_loop`, so the loop always terminates even if the model never produces the completion phrase.

Here is the full shape of the system you are about to run:

```
master_orchestrator_agent
├── full_plan_workflow_tool (AgentTool)
│   └── full_plan_workflow (SequentialAgent)
│       ├── intake_agent
│       ├── initial_planning_workflow (SequentialAgent)
│       │   ├── parallel_logistics_scout (ParallelAgent)
│       │   │   ├── venue_scout_agent
│       │   │   ├── catering_scout_agent
│       │   │   └── entertainment_scout_agent
│       │   └── initial_plan_synthesizer_agent
│       ├── budget_optimizer_workflow (SequentialAgent)
│       │   └── budget_refinement_loop (LoopAgent)
│       │       ├── accountant_agent
│       │       └── cost_cutter_agent
│       └── final_report_agent
├── guest_list_manager_tool (AgentTool)
│   └── guest_management_agent
└── communications_tool (AgentTool)
    └── communications_agent
```

> **Note on deprecation warnings.** When you run this cell you will see `DeprecationWarning` messages for `SequentialAgent`, `ParallelAgent`, and `LoopAgent`. ADK 2.6 introduced a new `Workflow` graph API as their eventual replacement. However, the ADK documentation explicitly states that `Workflow` **cannot yet be used as an `LlmAgent` sub-agent** — which is exactly how this capstone's orchestrator consumes these workflow agents via `AgentTool`. Migrating now would require restructuring the orchestrator in ways that go beyond the scope of this section. These warnings are safe to ignore. Once ADK lifts that restriction, the migration becomes a clean swap and will be covered in a future course update.

In [ ]:
from google.adk.agents import ParallelAgent, SequentialAgent, LoopAgent

parallel_logistics_scout = ParallelAgent(
    name="parallel_logistics_scout",
    sub_agents=[venue_scout_agent, catering_scout_agent, entertainment_scout_agent]
)

initial_planning_workflow = SequentialAgent(
    name="initial_planning_workflow",
    sub_agents=[parallel_logistics_scout, initial_plan_synthesizer_agent]
)

budget_refinement_loop = LoopAgent(
    name="budget_refinement_loop",
    sub_agents=[accountant_agent, cost_cutter_agent],
    max_iterations=3
)

budget_optimizer_workflow = SequentialAgent(
    name="budget_optimizer_workflow",
    sub_agents=[budget_refinement_loop]
)

full_plan_workflow = SequentialAgent(
    name="full_plan_workflow",
    sub_agents=[
        intake_agent,
        initial_planning_workflow,
        budget_optimizer_workflow,
        final_report_agent
    ]
)

/tmp/ipykernel_3442/2137418663.py:3: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_logistics_scout = ParallelAgent(
/tmp/ipykernel_3442/2137418663.py:8: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  initial_planning_workflow = SequentialAgent(
/tmp/ipykernel_3442/2137418663.py:13: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  budget_refinement_loop = LoopAgent(
/tmp/ipykernel_3442/2137418663.py:19: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  budget_optimizer_workflow = SequentialAgent(
/tmp/ipykernel_344

## Cell 9 — Define the Master Orchestrator

`master_orchestrator_agent` is the single entry point a user actually talks to. It never plans, sources venues, or drafts email itself. Its only job is to read the user's request and delegate to the right tool: `full_plan_workflow_tool` for planning, `guest_list_manager_tool` for guest management, `communications_tool` for drafting, and the plain `send_mock_email` function for sending. Everything underneath it — all ten specialist agents and five workflow agents — is invisible to the user.

In [ ]:
full_plan_workflow_tool = AgentTool(agent=full_plan_workflow)
guest_list_manager_tool = AgentTool(agent=guest_management_agent)
communications_tool = AgentTool(agent=communications_agent)

master_orchestrator_agent = Agent(
    name="master_orchestrator_agent",
    model=MODEL_NAME,
    instruction="Delegate: full_plan_workflow_tool for planning, guest_list_manager_tool for guests, communications_tool for email drafting, send_mock_email for sending.",
    tools=[
        full_plan_workflow_tool,
        guest_list_manager_tool,
        communications_tool,
        send_mock_email
    ]
)

## Cell 10 — Session Service and the Run Helper

`InMemorySessionService` holds this conversation in process memory for the lifetime of this notebook run. Nothing here persists once the runtime restarts — which is fine for this demo but not for production use. Later sections in this course swap in a persistent session service without changing any agent code.

`run_agent_query` wraps the standard ADK run loop. It creates a `Runner` for the given agent, drives `runner.run_async()` inside an `async for event in ...` loop, and returns the final text response once `event.is_final_response()` is `True`. Every turn in the conversation below calls this same helper.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

session_service = InMemorySessionService()


async def run_agent_query(agent, query, session, user_id):
    """Initializes a runner and executes a query for a given agent and session."""
    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )
    final_response = None
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(parts=[Part(text=query)], role="user"),
    ):
        if event.is_final_response():
            final_response = event.content.parts[0].text
    return final_response

## Cell 11 — Turn 1: Create the Session and Plan the Event

This cell creates the session under `master_orchestrator_agent`'s app name and fires the first user request. A single message is about to trigger `intake_agent`, all three scouts in parallel, the plan synthesizer, the entire budget refinement loop, and `final_report_agent` — all without you seeing any of it happen.

Run this cell and read through the response once it finishes. Then ask yourself: which agents ran? In what order? How many tool calls were made? How many loop iterations happened? You cannot answer any of these questions from the output alone. That is what we fix starting in the next lecture.

In [ ]:
USER_ID = "demo_user"

session = await session_service.create_session(
    app_name=master_orchestrator_agent.name,
    user_id=USER_ID
)

response_1 = await run_agent_query(
    master_orchestrator_agent,
    "Plan a 50-person AI tech meetup in Austin, TX with a $4,000 budget.",
    session,
    USER_ID
)
print(response_1)

/usr/local/lib/python3.13/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


Here is the complete event plan for your **50-person AI Tech Meetup in Austin, TX**:

---

# 📋 Austin AI Tech Meetup Plan

## 1. Executive Summary
- **Event Type:** AI Tech Meetup & Networking Mixer
- **Target Attendance:** 50 Guests
- **Location:** Austin, TX
- **Total Budget:** $4,000.00
- **Total Estimated Cost:** **$2,350.00**
- **Remaining Surplus / Contingency:** **+$1,650.00** *(41.25% under budget)*

---

## 2. Selected Vendors & Components

### 🏢 Venue: Fuse Workspace (East MLK / Rally Room)
- **Location:** 2112 E MLK Jr Blvd, East Austin, TX
- **Cost:** **$700.00** *(4-hour rental @ $175/hr)*
- **Highlights & Amenities:**
  - Capacity up to 50 guests
  - Screen-share display technology & HD webcam support
  - Wireless microphones for speakers and Q&A sessions
  - Free on-site parking for all attendees
  - On-site concierge and kitchenette access

### 🌮 Catering: Tacodeli Catering
- **Cuisine:** Artisanal Tex-Mex Build-Your-Own Taco Bar
- **Cost:** **$1,150.00** *(~$18.50/pers

## Cell 12 — Turn 2: Add a Guest

This turn routes through `guest_list_manager_tool` to `guest_management_agent`, completely independent of the planning pipeline that just ran. Run it and confirm the guest was added.

In [ ]:
response_2 = await run_agent_query(
    master_orchestrator_agent,
    "Add Grace Hopper to the guest list. Her email is grace.hopper@example.com.",
    session,
    USER_ID
)
print(response_2)

**Grace Hopper** (`grace.hopper@example.com`) has been successfully added to the guest list!


## Cell 13 — Turn 3: Draft the Announcement Email

This turn calls `communications_tool`, which runs `communications_agent` and validates its output against the `EmailDraft` schema defined in Cell 7. Run it and look at the drafted email.

In [ ]:
response_3 = await run_agent_query(
    master_orchestrator_agent,
    "Draft an announcement email for this event.",
    session,
    USER_ID
)
print(response_3)

Here is the drafted announcement email for the event:

---

**Subject:** You're Invited: Austin AI Tech Meetup, Tacos & Trivia at Fuse Workspace!

**Body:**
> Join us for an exciting evening of networking, innovation, and fun at the Austin AI Tech Meetup! We are bringing together 50 tech enthusiasts, founders, and AI builders for a night packed with great conversations, delicious food, and friendly competition.
>
> 📍 **Venue:** Fuse Workspace (Rally Room), 2112 E MLK Jr Blvd, East Austin, TX  
> 🌮 **Food & Drink:** Full artisanal build-your-own taco bar catered by Austin favorite, Tacodeli  
> 🧠 **Entertainment:** Custom AI & Tech Trivia Show featuring digital scoring and prizes!  
>
> Space is strictly limited to 50 attendees, so please RSVP early to guarantee your spot.
>
> We can't wait to see you there!

---

Would you like to make any edits or proceed with sending a mock announcement email to the guest list?


## Cell 14 — Turn 4: Send the Email

This turn calls `send_mock_email` directly — no sub-agent involved. Run it and look at the confirmation. The system has now completed a full four-turn event planning workflow with zero observation code anywhere in it.

In [ ]:
response_4 = await run_agent_query(
    master_orchestrator_agent,
    "Send the email to everyone on the guest list.",
    session,
    USER_ID
)
print(response_4)

The announcement email has been successfully sent!

### **Email Send Summary:**
- **Status:** Sent
- **Subject:** *You're Invited: Austin AI Tech Meetup, Tacos & Trivia at Fuse Workspace!*
- **Recipients Reached:** 1 (Grace Hopper – `grace.hopper@example.com`)

Let me know if you need to add more guests, modify the event details, or handle any other planning tasks!
